# Atmospheric-DA impact on the land state

This notebook connects the atmospheric DA experiments to the land-state evaluation. For S0 it compares the land-state error in DART10, DART20, and DART40 with CTRL10. The DART10–CTRL10 pair has the same ensemble size and is the primary atmospheric-DA attribution; DART20 and DART40 additionally show ensemble-size sensitivity.

The figure answers three questions: where does DA reduce absolute error, at which latitudes does RMSE improve, and how large is the global benefit? Positive values consistently mean improvement. S1 contains DART40-S1 only, so it can be evaluated absolutely in notebook 01 but cannot support a DA-minus-control impact figure.

## 1. Libraries and project setup

In [ ]:
from pathlib import Path
import sys
import xarray as xr

WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
DIAGNOSTIC_DIR = WORK_DIR / "diagnostic"
if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

from configs.output_paths import workflow_output_dirs
from util.model_evaluation import evaluation_cache_path
from util.initial_lnd_da_impact import (
    derive_land_da_impact_from_evaluations,
    plot_land_da_impact,
    save_impact_products,
)

## 2. User setup

In [ ]:
EXPERIMENTS_BY_SEASON = {
    "S0": ["CTRL10-S0", "DART10-S0", "DART20-S0", "DART40-S0"],
    "S1": ["DART40-S1"],
}
PERIOD_BY_SEASON = {"S0": "201112-201112", "S1": "201205-201205"}

SEASON = "S0"
VARIABLE = "H2OSOI"
BASELINE = "CTRL10-S0"
COMPARISONS = ("DART10-S0", "DART20-S0", "DART40-S0")
PRIMARY_EXPERIMENT = "DART10-S0"
SHOW = True
FIGURE_DPI = 200

LABELS = {
    "CTRL10-S0": "CTRL10 (no DA)",
    "DART10-S0": "DART10",
    "DART20-S0": "DART20",
    "DART40-S0": "DART40",
    "DART40-S1": "DART40-S1",
}
COLORS = {
    "DART10-S0": "#e69f00",
    "DART20-S0": "#009e73",
    "DART40-S0": "#d55e00",
}

## 3. Load the land-evaluation caches

Run `01_compute_and_plot_bias_metrics.ipynb` with `SEASON = "S0"` first if these caches do not yet exist.

In [ ]:
if SEASON != "S0":
    raise ValueError(
        "An impact figure requires a no-DA baseline. S1 has DART40-S1 only; "
        "use notebook 01 for its absolute evaluation."
    )

workflow_data_dir, workflow_figure_dir = workflow_output_dirs(
    "analysis_lnd_init", output_root=DIAGNOSTIC_OUTPUT_ROOT
)
cache_dir = workflow_data_dir / "bias_metrics"
output_data_dir = workflow_data_dir / "atm_da_land_state_impact"
output_figure_dir = workflow_figure_dir / "atm_da_land_state_impact"
period = PERIOD_BY_SEASON[SEASON]
requested = (BASELINE, *COMPARISONS)

paths = {
    experiment: evaluation_cache_path(cache_dir, experiment, VARIABLE, period)
    for experiment in requested
}
missing = [path for path in paths.values() if not path.is_file()]
if missing:
    formatted = "\n".join(f"  - {path}" for path in missing)
    raise FileNotFoundError(
        "Missing land-evaluation caches. Run notebook 01 for S0 first:\n" + formatted
    )
evaluations = {name: xr.load_dataset(path) for name, path in paths.items()}
paths

## 4. Derive and save the impact diagnostics

In [ ]:
impact = derive_land_da_impact_from_evaluations(
    evaluations, baseline=BASELINE, experiments=COMPARISONS
)
stem = f"{SEASON}_{VARIABLE}_atm_da_land_state_impact"
summary = save_impact_products(
    impact,
    data_path=output_data_dir / f"{stem}.nc",
    csv_path=output_data_dir / f"{stem}.csv",
)
summary

## 5. Plot the attribution figure

In [ ]:
figure = plot_land_da_impact(
    impact,
    primary_experiment=PRIMARY_EXPERIMENT,
    experiment_labels=LABELS,
    colors=COLORS,
)
output_figure_dir.mkdir(parents=True, exist_ok=True)
png_path = output_figure_dir / f"{stem}.png"
pdf_path = output_figure_dir / f"{stem}.pdf"
figure.savefig(png_path, dpi=FIGURE_DPI, bbox_inches="tight")
figure.savefig(pdf_path, bbox_inches="tight")
print(f"[FIGURE] {png_path}")
print(f"[FIGURE] {pdf_path}")
if SHOW:
    import matplotlib.pyplot as plt
    plt.show()